# LHRM Author-Aligned Model Recovery & Full Evaluation (AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A)

**Client-Ready Author-Code-Aligned Implementation (15 Epochs, Full AAPD Training, Non-Leaky HRRM Pipeline & Client Demo)**

### Key Technical Characteristics:
1. **Architecture (`R_AUTHOR`)**: Directly implements the original authors' reference architecture from `Model/lhrm.py:144-192`:
   - Document representation: `bert(...).pooler_output` $\in \mathbb{R}^{B \times 768}$ (`Model/lhrm.py:185`).
   - Label representation: Trainable $\text{nn.Parameter}$ initialized once from pre-trained BERT token means (`Model/lhrm.py:147, 222-243`).
   - Label self-attention: `nn.MultiheadAttention(768, heads=4)` with residual addition and LayerNorm (`Model/lhrm.py:150-158`).
   - Similarity: Unscaled dot-product attention $w = \text{softmax}(h_\text{doc} \cdot L^T, \text{dim}=1)$ (`Model/lhrm.py:159`).
   - Aggregation: $\text{agg} = w \cdot L \in \mathbb{R}^{B \times 768}$ (`Model/lhrm.py:160`).
   - Concatenation: $\text{combined} = [h_\text{doc}, \text{agg}] \in \mathbb{R}^{B \times 1536}$ (`Model/lhrm.py:190`).
   - Classification: $\text{nn.Linear}(1536, 54)$ (`Model/lhrm.py:182, 191`).
2. **Objective**: Standard `torch.nn.BCEWithLogitsLoss()` matching the authors' codebase (`Model/lhrm.py:339`) and the successful DR2 control.
3. **Training Protocol**: 15 epochs, batch size 8, learning rate 2e-5, weight decay 0.01, AdamW optimizer, seed 42.
4. **Step-300 Informational Checkpoint**: Informational early progress check on the fixed 200 diagnostic documents to monitor loss and ranking behavior without aborting the full 15-epoch training run.
5. **Data Quarantine**: **Test split is strictly quarantined.** Only AAPD Train (53,840) and Dev (1,000) splits are used.
6. **Non-Leaky Dev Partition & Checkpoint Selection**: Deterministic 4-way Dev partition:
   - 200 docs: Continuity diagnostic check (`diag_ids`, matching registered RUN_0 sample).
   - 400 docs: Neural model checkpoint selection (`model_sel_ids`, exclusively drives best checkpoint selection).
   - 200 docs: HRRM regressor training (`hrrm_fit_ids`).
   - 200 docs: Held-out HRRM vs Neural comparison (`hrrm_eval_ids`).
7. **Consistent Selected Checkpoint**: The best neural checkpoint selected on the 400-doc subset is loaded and used consistently for HRRM fitting, HRRM held-out evaluation, and client inference.
8. **Client Demonstration**: Interactive inference on scientific text/abstracts producing Top-5 predictions with confidence scores and unverified label mapping disclosure.
9. **Bounded Disk Management**: Checkpoints are strictly bounded to `latest.pt`, `previous.pt`, `best_dev_model_only.pt`, and `final_model_only.pt` (~3.5 GB peak disk usage).

- **Run ID:** `AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A`
- **Bundle Archive:** `lhrm_gpu_bundle_r_author_20261015.zip` (SHA-256: `e25c583e76d44873e0bdeebcaa99973db8ac4763b79023afd2787902a8e65bbf`)
- **Internal Manifest SHA-256:** `8a6ebf146293e22e3a597f992924c84dd06ffc4fe5276d9978add3dfdd9fe9ce`
- **Sidecar Manifest SHA-256:** `7a5f5fa02331954817917569a3eff5166289801e7a62c8b21a4d75bf9ce1c1bf`
- **Label Order Hash:** `a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b`

In [ ]:
# Cell 1: Canonical Configuration, Seed Initialization & Official BERT Acquisition
import os, sys, time, json, hashlib, shutil, gc, random
from pathlib import Path
import numpy as np
import torch

print("=== LHRM R_AUTHOR Recovery: Configuration & Environment Preflight ===", flush=True)

# ── Canonical Configuration (Patch 3) ──
RUN_ID = "AAPD_R_AUTHOR_BCE_RECOVERY_20261015_A"
SEED = 42
MODEL_VARIANT = "R_AUTHOR"
LOSS_TYPE = "bce"
LR = 2e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE = 8
MAX_LEN = 512
NUM_EPOCHS = 15
EXPECTED_LABEL_ORDER_HASH = "a3d662f3920259a94ed61d83f98f21cabe7f908d98a5662ccd9adc2413cc000b"
RESUME_CHECKPOINT_PATH = None  # Explicit path string (e.g. "/kaggle/input/.../latest.pt") or None for fresh run

# ── Seed Initialization (Patch 4 / A4) ──
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Random seeds initialized to {SEED} across Python, NumPy, PyTorch CPU & CUDA.")

# ── GPU Verification ──
if not torch.cuda.is_available():
    raise RuntimeError("GPU GATE: CUDA GPU required — select Kaggle GPU runtime (T4 x2 or P100)")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, " | CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), flush=True)

import sklearn, transformers
print(f"Transformers: {transformers.__version__} | Scikit-Learn: {sklearn.__version__}")

# ── Acquire Official Pinned BERT Backbone ──
PINNED_COMMIT = "86b5e0934494bd15c9632b12f734a8a67f723594"
REPO = "google-bert/bert-base-uncased"
BERT_DIR = Path("/kaggle/working/google_bert_bert_base_uncased_official")
SOURCE_META = Path("/kaggle/working/OFFICIAL_BERT_SOURCE.json")
WEIGHTS = ("model.safetensors", "pytorch_model.bin")

def verify_and_record():
    if not BERT_DIR.exists():
        return False
    present = [f for f in WEIGHTS if (BERT_DIR / f).exists()]
    if not present or not (BERT_DIR / "config.json").exists():
        return False
    if not SOURCE_META.exists():
        meta = {
            "source_repo": REPO,
            "pinned_commit": PINNED_COMMIT,
            "weight_files": present,
            "verified_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        SOURCE_META.write_text(json.dumps(meta, indent=2))
    return True

if not verify_and_record():
    print(f"Acquiring official BERT backbone ({REPO} @ {PINNED_COMMIT})...")
    attached_dirs = [
        Path("/kaggle/input/bert-base-uncased"),
        Path("/kaggle/input/google-bert-bert-base-uncased"),
        Path("/kaggle/input/bert-base-uncased-official"),
    ]
    found_attached = False
    for ad in attached_dirs:
        if ad.exists() and any((ad / w).exists() for w in WEIGHTS) and (ad / "config.json").exists():
            print(f"Using attached BERT from: {ad}")
            shutil.copytree(ad, BERT_DIR, dirs_exist_ok=True)
            found_attached = True
            break
            
    if not found_attached:
        from huggingface_hub import snapshot_download
        print(f"Downloading from HuggingFace Hub: {REPO} revision {PINNED_COMMIT}...")
        snapshot_download(
            repo_id=REPO,
            revision=PINNED_COMMIT,
            local_dir=str(BERT_DIR),
            local_dir_use_symlinks=False,
            ignore_patterns=["*.msgpack", "*.h5", "*.ot"]
        )

if not verify_and_record():
    raise RuntimeError("Failed to verify official BERT backbone files")
print(f"Official BERT ready at: {BERT_DIR}", flush=True)

In [ ]:
# Cell 2: Code Bundle Verification (Dual-Path: Raw ZIP or Auto-Extracted Files)
import zipfile

BUNDLE_NAME = "lhrm_gpu_bundle_r_author_20261015.zip"
EXPECTED_BUNDLE_SHA256 = "e25c583e76d44873e0bdeebcaa99973db8ac4763b79023afd2787902a8e65bbf"
EXPECTED_INTERNAL_MANIFEST_SHA256 = "8a6ebf146293e22e3a597f992924c84dd06ffc4fe5276d9978add3dfdd9fe9ce"
EXPECTED_SIDECAR_MANIFEST_SHA256 = "7a5f5fa02331954817917569a3eff5166289801e7a62c8b21a4d75bf9ce1c1bf"

print(f"=== Locating and Verifying Code Bundle: {BUNDLE_NAME} ===", flush=True)

# Path 1: Check for raw uploaded ZIP archive
search_locations = [
    Path(f"/kaggle/input/{BUNDLE_NAME}"),
    Path(f"/kaggle/input/lhrm-gpu-bundle/{BUNDLE_NAME}"),
    Path(f"/kaggle/input/lhrm-gpu-bundle-r-author-20261015/{BUNDLE_NAME}"),
    Path(f"/kaggle/input/lhrm-r-author-bundle/{BUNDLE_NAME}"),
    Path(f"/kaggle/working/{BUNDLE_NAME}"),
]

for d in Path("/kaggle/input").glob("*"):
    if d.is_dir():
        cand = d / BUNDLE_NAME
        if cand.exists() and cand not in search_locations:
            search_locations.append(cand)

bundle_path = None
for p in search_locations:
    if p.exists() and p.is_file():
        bundle_path = p
        break

if bundle_path:
    print(f"[PATH 1: RAW ARCHIVE] Found bundle ZIP at: {bundle_path}")
    actual_hash = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
    print(f"Archive SHA-256: {actual_hash}")
    assert actual_hash == EXPECTED_BUNDLE_SHA256, (
        f"BUNDLE HASH MISMATCH! Expected {EXPECTED_BUNDLE_SHA256}, got {actual_hash}"
    )
    with zipfile.ZipFile(bundle_path, "r") as z:
        z.extractall("/kaggle/working")
    print("Extracted bundle archive successfully into /kaggle/working.")
    
    # Verify internal manifest inside extracted archive
    internal_man = Path("/kaggle/working/BUNDLE_MANIFEST.json")
    assert internal_man.exists(), "Extracted bundle is missing internal BUNDLE_MANIFEST.json"
    int_hash = hashlib.sha256(internal_man.read_bytes()).hexdigest()
    assert int_hash == EXPECTED_INTERNAL_MANIFEST_SHA256, (
        f"Internal manifest hash mismatch! Expected {EXPECTED_INTERNAL_MANIFEST_SHA256}, got {int_hash}"
    )
    print("Verified internal manifest SHA-256 successfully.")
else:
    print("[PATH 2: AUTO-EXTRACTED] Archive ZIP not found directly; searching for auto-extracted bundle files...")
    manifest_candidates = list(Path("/kaggle/input").rglob("BUNDLE_MANIFEST.json"))
    matched = None
    for mc in manifest_candidates:
        m_hash = hashlib.sha256(mc.read_bytes()).hexdigest()
        if m_hash == EXPECTED_INTERNAL_MANIFEST_SHA256:
            matched = mc
            print(f"Matched authentic internal manifest at: {mc} (SHA: {m_hash})")
            break
            
    assert matched is not None, (
        f"Could not locate {BUNDLE_NAME} or extracted BUNDLE_MANIFEST.json with internal SHA {EXPECTED_INTERNAL_MANIFEST_SHA256}. "
        f"Found candidates: {[(str(c), hashlib.sha256(c.read_bytes()).hexdigest()[:12]) for c in manifest_candidates]}"
    )
    
    # Verify all 16 constituent files against the internal manifest
    source_dir = matched.parent
    manifest_data = json.loads(matched.read_text(encoding="utf-8"))
    files_dict = manifest_data.get("files", {})
    print(f"Verifying {len(files_dict)} constituent files from {source_dir} before staging...")
    
    for rel_path, expected_file_sha in files_dict.items():
        src_file = source_dir / rel_path
        assert src_file.exists(), f"Constituent bundle file missing from auto-extracted dataset: {src_file}"
        actual_file_sha = hashlib.sha256(src_file.read_bytes()).hexdigest()
        assert actual_file_sha == expected_file_sha, (
            f"Constituent file hash mismatch for {rel_path}! Expected {expected_file_sha}, got {actual_file_sha}"
        )
        dest = Path("/kaggle/working") / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_file, dest)
        
    # Also copy manifest itself and SHA256SUMS.txt if present
    shutil.copy2(matched, Path("/kaggle/working/BUNDLE_MANIFEST.json"))
    if (source_dir / "SHA256SUMS.txt").exists():
        shutil.copy2(source_dir / "SHA256SUMS.txt", Path("/kaggle/working/SHA256SUMS.txt"))
    print(f"Successfully verified and staged all {len(files_dict)} bundle files into /kaggle/working.")

# Ensure working directory is on sys.path
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

from src.lhrm_baseline.model import LHRMModel, CSAM_R_AUTHOR, build_author_label_embeddings
from src.lhrm_baseline.losses import get_loss
from src.lhrm_baseline.hrrm import HRRM, create_dev_partition, evaluate_hrrm_heldout
from src.lhrm_baseline.inference import predict_abstract, format_prediction_report
print("Successfully imported verified LHRM baseline modules.", flush=True)

In [ ]:
# Cell 3: Authentic AAPD Dataset Loading, Validation & Test Quarantine
import pandas as pd
import zipfile

print("=== AAPD Data Preparation & Strict Quarantine Verification ===", flush=True)

DATA_DIR = Path("/kaggle/working/data/processed/AAPD")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Locate authentic AAPD data files ──
train_csv_path = None
dev_csv_path = None

# Search for direct CSV files under /kaggle/input
for cand_train in Path("/kaggle/input").rglob("train.csv"):
    parent = cand_train.parent
    if (parent / "dev.csv").exists():
        train_csv_path = cand_train
        dev_csv_path = parent / "dev.csv"
        print(f"Found authentic AAPD CSVs at: {parent}")
        break

# Fallback: search for AAPD.zip archive
if not train_csv_path:
    for cand_zip in Path("/kaggle/input").rglob("*.zip"):
        if "aapd" in cand_zip.name.lower():
            print(f"Found candidate AAPD archive: {cand_zip}")
            with zipfile.ZipFile(cand_zip, "r") as z:
                names = z.namelist()
                has_train = any("train.csv" in n for n in names)
                has_dev = any("dev.csv" in n for n in names)
                if has_train and has_dev:
                    extract_dest = Path("/kaggle/working/data/raw/AAPD")
                    extract_dest.mkdir(parents=True, exist_ok=True)
                    z.extractall(extract_dest)
                    for extracted_tr in extract_dest.rglob("train.csv"):
                        if (extracted_tr.parent / "dev.csv").exists():
                            train_csv_path = extracted_tr
                            dev_csv_path = extracted_tr.parent / "dev.csv"
                            print(f"Extracted AAPD CSVs from archive to: {extracted_tr.parent}")
                            break
            if train_csv_path:
                break

assert train_csv_path is not None and dev_csv_path is not None, (
    "FATAL: Could not locate authentic AAPD train.csv and dev.csv in /kaggle/input (direct or inside ZIP)"
)

# ── Load authentic CSVs ──
df_train = pd.read_csv(train_csv_path)
df_dev = pd.read_csv(dev_csv_path)

print(f"Raw Train CSV shape: {df_train.shape} | Raw Dev CSV shape: {df_dev.shape}")

# ── Identify Document Text Column (Patch 2) ──
# Authentic AAPD header: 'abstract,cmp-lg,cond-mat.dis-nn,...'
text_candidates = ["abstract", "text"]
text_col = next((c for c in text_candidates if c in df_train.columns), None)
assert text_col is not None, (
    f"FATAL: Document text column not found in Train CSV! Columns: {list(df_train.columns[:5])}"
)
print(f"Using document text column: '{text_col}'")

# ── Extract and Validate 54 Target Columns ──
metadata_cols = {"abstract", "text", "doc_id", "id", "ID"}
label_cols = [c for c in df_train.columns if c not in metadata_cols]
dev_label_cols = [c for c in df_dev.columns if c not in metadata_cols]

assert len(label_cols) == 54, f"Expected exactly 54 label columns, got {len(label_cols)}"
assert len(dev_label_cols) == 54, f"Expected exactly 54 Dev label columns, got {len(dev_label_cols)}"
assert label_cols == dev_label_cols, "Train and Dev target columns or order do not match identically!"

# ── Verify Registered Label-Order Hash ──
label_order_hash = hashlib.sha256(json.dumps(label_cols).encode()).hexdigest()
print(f"Observed label-order hash: {label_order_hash}")
assert label_order_hash == EXPECTED_LABEL_ORDER_HASH, (
    f"Label order hash mismatch! Expected {EXPECTED_LABEL_ORDER_HASH}, got {label_order_hash}"
)

# ── Verify Document Counts ──
train_texts = df_train[text_col].astype(str).tolist()
train_labels = df_train[label_cols].values.astype(np.float32)

dev_texts = df_dev[text_col].astype(str).tolist()
dev_labels = df_dev[label_cols].values.astype(np.float32)

print(f"Train documents: {len(train_texts):,} | Dev documents: {len(dev_texts):,}")
assert len(train_texts) == 53840, f"Expected 53,840 train documents, got {len(train_texts)}"
assert len(dev_texts) == 1000, f"Expected 1,000 dev documents, got {len(dev_texts)}"

# ── Strict Quarantine Check: Test split must NEVER be loaded ──
assert "test_texts" not in locals(), "Quarantine violation: test_texts must not exist"
assert "test_labels" not in locals(), "Quarantine violation: test_labels must not exist"
print("Test split quarantine confirmed: Test split is strictly unread.", flush=True)

# ── Save processed representations for reproducibility ──
from src.lhrm_baseline.data import save_processed, MultiLabelDataset, collate_fn
save_processed(str(DATA_DIR), "train", train_texts, train_labels, label_cols)
save_processed(str(DATA_DIR), "dev", dev_texts, dev_labels, label_cols)
with open(DATA_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(label_cols, f, indent=2)

print("AAPD processed splits saved successfully.", flush=True)

In [ ]:
# Cell 4: Author Label Embeddings Precomputation & R_AUTHOR Architecture Initialization
from transformers import BertModel, BertTokenizer
from src.lhrm_baseline.model import build_bert_model, build_author_label_embeddings

print("=== Initializing Author-Aligned R_AUTHOR Architecture ===", flush=True)

# Re-seed to guarantee identical deterministic parameter initialization
set_seed(SEED)

device = torch.device("cuda")
tokenizer = BertTokenizer.from_pretrained(str(BERT_DIR))
bert = build_bert_model(str(BERT_DIR), device=device)

defs_path = Path("/kaggle/working/Label_Definition/arxiv_labels_definitions.json")
assert defs_path.exists(), f"Missing label definitions at {defs_path}"

# Build initial author label embeddings from BERT token means (Model/lhrm.py:222-243)
print("Precomputing initial label embeddings from BERT token means over label definition strings...")
label_emb = build_author_label_embeddings(
    label_list=label_cols,
    tokenizer=tokenizer,
    bert=bert,
    defs_path=str(defs_path),
    max_len=64,
    device=device,
)
print(f"Precomputed label embedding tensor shape: {tuple(label_emb.shape)}")
assert label_emb.shape == (54, 768)

# Construct R_AUTHOR model with trainable nn.Parameter label embeddings
model = LHRMModel(
    bert_model=bert,
    num_labels=len(label_cols),
    variant="R_AUTHOR",
    label_emb=label_emb,
).to(device)

# Forward pass sanity check on dummy batch
dummy_ids = torch.randint(0, 1000, (2, 64), device=device)
dummy_mask = torch.ones_like(dummy_ids, device=device)
with torch.no_grad():
    test_logits, test_diag = model(dummy_ids, dummy_mask, return_alpha=True)

print(f"Sanity logits shape: {tuple(test_logits.shape)} (expected: [2, 54])")
print(f"Sanity alpha shape:  {tuple(test_diag['alpha'].shape)} (expected: [2, 54])")
assert test_logits.shape == (2, 54)
assert test_diag["alpha"].shape == (2, 54)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print("R_AUTHOR architecture initialization verified successfully.", flush=True)

In [ ]:
# Cell 5: Full 15-Epoch Training Loop with Real Resume, Disk Management & Safe Selection
from torch.utils.data import DataLoader
from src.lhrm_baseline.train import predict_epoch_loader, save_predictions
from src.lhrm_baseline.metrics import compute_all_metrics

EXP_DIR = Path(f"/kaggle/working/experiments/{RUN_ID}")
CKPT_DIR = EXP_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Disk Space Preflight & Bounded Storage Check (Bug 3) ──
total_b, used_b, free_b = shutil.disk_usage("/kaggle/working")
free_gb = free_b / (1024**3)
print(f"Disk check: {free_gb:.1f} GB available in /kaggle/working.")
print("Bounded checkpoint strategy: keeping only latest.pt, previous.pt, best_dev_model_only.pt, and final_model_only.pt (~3.5 GB peak).")
if free_gb < 4.0:
    print(f"WARNING: Available disk space ({free_gb:.1f} GB) is low. Monitoring closely.")

# ── Deterministic 4-Way Dev Partition (Patch 6) ──
dev_partition = create_dev_partition(n_docs=len(dev_texts), seed=SEED)
diag_ids = dev_partition["diagnostic_ids"]              # 200 docs (continuity check)
model_sel_ids = dev_partition["model_selection_ids"]    # 400 docs (neural model selection)
hrrm_fit_ids = dev_partition["hrrm_fit_ids"]            # 200 docs (HRRM training)
hrrm_eval_ids = dev_partition["hrrm_eval_ids"]          # 200 docs (held-out HRRM evaluation)

# Validate exact disjoint partition coverage
assert len(diag_ids) == 200 and len(model_sel_ids) == 400 and len(hrrm_fit_ids) == 200 and len(hrrm_eval_ids) == 200
all_partition_ids = set(diag_ids) | set(model_sel_ids) | set(hrrm_fit_ids) | set(hrrm_eval_ids)
assert len(all_partition_ids) == 1000, "Partition is not mutually exclusive or does not cover all 1,000 Dev docs!"
assert len(set(diag_ids) & set(model_sel_ids)) == 0, "Diagnostic and Selection sets overlap!"
assert len(set(model_sel_ids) & set(hrrm_eval_ids)) == 0, "Selection and HRRM eval sets overlap!"

# Save partition metadata and checksum
partition_meta_path = EXP_DIR / "dev_partition.json"
with open(partition_meta_path, "w", encoding="utf-8") as pf:
    json.dump({
        "n_total": 1000,
        "seed": SEED,
        "diagnostic_ids": diag_ids,
        "model_selection_ids": model_sel_ids,
        "hrrm_fit_ids": hrrm_fit_ids,
        "hrrm_eval_ids": hrrm_eval_ids,
    }, pf, indent=2)
partition_checksum = hashlib.sha256(partition_meta_path.read_bytes()).hexdigest()
print(f"Saved Dev 4-way partition to {partition_meta_path} (SHA-256: {partition_checksum[:12]}...)")

# ── Datasets & Loaders ──
train_ds = MultiLabelDataset(train_texts, train_labels, tokenizer, max_len=MAX_LEN)
dev_ds   = MultiLabelDataset(dev_texts, dev_labels, tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

# ── Loss & Optimizer ──
criterion = get_loss(LOSS_TYPE)  # BCEWithLogitsLoss
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda")

# ── State Tracking & Fail-Closed Resume Handling (Bug 2 & Patch 5) ──
start_epoch = 1
total_steps = 0
steps_per_epoch = len(train_loader)
total_global_steps = NUM_EPOCHS * steps_per_epoch
history = []
best_sel_p1 = -1.0
best_epoch = 0
best_model_sd = None
step_300_reported = False

metrics_csv_path = EXP_DIR / "epoch_metrics.csv"
history_json_path = EXP_DIR / "train_history.json"

if RESUME_CHECKPOINT_PATH is not None:
    resume_path = Path(RESUME_CHECKPOINT_PATH)
    if not resume_path.exists():
        raise FileNotFoundError(
            f"FAIL-CLOSED GATE: Resume requested with RESUME_CHECKPOINT_PATH='{RESUME_CHECKPOINT_PATH}', "
            "but the file does not exist! Refusing to start an unverified fresh run."
        )
    print(f"\n=== Loading Verified Resume Checkpoint from: {resume_path} ===")
    ckpt = torch.load(str(resume_path), map_location=device, weights_only=False)
    
    assert ckpt.get("variant") == "R_AUTHOR", f"Variant mismatch: {ckpt.get('variant')}"
    if "label_order_hash" in ckpt:
        assert ckpt["label_order_hash"] == EXPECTED_LABEL_ORDER_HASH, "Label order mismatch in resume checkpoint!"
        
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scaler and "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
        
    start_epoch = ckpt["epoch"] + 1
    total_steps = ckpt.get("step", (start_epoch - 1) * steps_per_epoch)
    history = ckpt.get("history", [])
    best_sel_p1 = ckpt.get("best_sel_p1", -1.0)
    best_epoch = ckpt.get("best_epoch", 0)
    step_300_reported = (total_steps >= 300)
    
    # Restore best selection model checkpoint (Bug 2)
    best_ckpt_target = CKPT_DIR / "best_dev_model_only.pt"
    if "best_model_state_dict" in ckpt:
        best_model_sd = ckpt["best_model_state_dict"]
        torch.save({
            "epoch": best_epoch,
            "step": ckpt.get("best_step", total_steps),
            "model_state_dict": best_model_sd,
            "variant": "R_AUTHOR",
            "selection_subset": "400_docs",
            "selection_P@1": best_sel_p1,
            "run_id": RUN_ID,
        }, best_ckpt_target)
        print(f"Restored best_dev_model_only.pt from checkpoint payload (Best Epoch {best_epoch}, Selection P@1: {best_sel_p1:.4f})")
    elif (resume_path.parent / "best_dev_model_only.pt").exists():
        shutil.copy2(resume_path.parent / "best_dev_model_only.pt", best_ckpt_target)
        best_sd_loaded = torch.load(best_ckpt_target, map_location="cpu", weights_only=False)
        best_model_sd = best_sd_loaded["model_state_dict"]
        print(f"Restored best_dev_model_only.pt from resume directory: {resume_path.parent}")
    else:
        raise RuntimeError(
            f"FAIL-CLOSED: Resume checkpoint does not contain best_model_state_dict and best_dev_model_only.pt was not found beside {resume_path}!"
        )
        
    # Restore or reconstruct step300_info.json (Bug 4)
    if step_300_reported and not (EXP_DIR / "step300_info.json").exists():
        cand_s300 = resume_path.parent / "step300_info.json"
        if not cand_s300.exists():
            cand_s300 = resume_path.parent.parent / "step300_info.json"
        if cand_s300.exists():
            shutil.copy2(cand_s300, EXP_DIR / "step300_info.json")
        else:
            s300_record = ckpt.get("step300_info", {
                "step": 300,
                "train_loss_step300": None,
                "diag_P@1": history[0].get("diag_P@1", None) if history else None,
                "prior_reference": 63,
                "note": "RESTORED_FROM_RESUME_CHECKPOINT"
            })
            with open(EXP_DIR / "step300_info.json", "w", encoding="utf-8") as sf:
                json.dump(s300_record, sf, indent=2)
                
    # Restore RNG states
    if "torch_rng_state" in ckpt:
        torch.set_rng_state(ckpt["torch_rng_state"])
    if "cuda_rng_state" in ckpt and ckpt["cuda_rng_state"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state(ckpt["cuda_rng_state"])
    if "numpy_rng_state" in ckpt:
        np.random.set_state(ckpt["numpy_rng_state"])
    if "python_rng_state" in ckpt:
        random.setstate(ckpt["python_rng_state"])
        
    print(f"Resumed successfully at start of Epoch {start_epoch} (total steps completed: {total_steps:,})")
else:
    # Initialize fresh CSV header
    with open(metrics_csv_path, "w") as f:
        f.write("epoch,train_loss,global_steps,val_P@1,val_P@3,val_P@5,val_nDCG@3,val_nDCG@5,val_micro_f1,val_macro_f1,sel_P@1,diag_P@1\n")

# Save resolved config
config_resolved = {
    "run_id": RUN_ID,
    "model_variant": MODEL_VARIANT,
    "loss": LOSS_TYPE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "batch_size": BATCH_SIZE,
    "max_len": MAX_LEN,
    "num_epochs": NUM_EPOCHS,
    "seed": SEED,
    "label_order_hash": EXPECTED_LABEL_ORDER_HASH,
    "steps_per_epoch": steps_per_epoch,
    "total_global_steps": total_global_steps,
    "start_epoch": start_epoch,
    "resumed_from": str(RESUME_CHECKPOINT_PATH) if RESUME_CHECKPOINT_PATH else None,
}
with open(EXP_DIR / "CONFIG_RESOLVED.json", "w", encoding="utf-8") as cf:
    json.dump(config_resolved, cf, indent=2)

print(f"=== Starting Training ({RUN_ID}) ===", flush=True)
print(f"Epochs: {start_epoch} to {NUM_EPOCHS} | Steps per Epoch: {steps_per_epoch} | Total Global Steps: {total_global_steps}")

t_train_start = time.time()

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    epoch_start_time = time.time()
    model.train()
    total_epoch_loss = 0.0
    
    for batch_idx, batch in enumerate(train_loader, 1):
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = batch["labels"].to(device)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            logits, _ = model(ids, mask)
            loss = criterion(logits, y)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loss_val = float(loss.item())
        total_epoch_loss += loss_val
        total_steps += 1
        
        # Step-300 informational check (detects catastrophic early breakdown)
        if total_steps == 300 and not step_300_reported:
            step_300_reported = True
            model.eval()
            with torch.no_grad():
                y_true_d, y_pred_d, _ = predict_epoch_loader(model, dev_loader, device, None, None)
            diag_metrics = compute_all_metrics(y_true_d[diag_ids], y_pred_d[diag_ids], [1, 3, 5], threshold=0.5)
            s300_info = {
                "step": 300,
                "train_loss_step300": loss_val,
                "diag_P@1": diag_metrics.get("P@1", 0.0),
                "diag_hits_at_1": int(round(diag_metrics.get("P@1", 0.0) * len(diag_ids))),
                "prior_reference": 63,
            }
            print(f"\n[STEP 300 CHECK] train_loss={loss_val:.4f} | diag_P@1={s300_info['diag_P@1']:.4f} ({s300_info['diag_hits_at_1']}/200 vs prior 63/200)\n", flush=True)
            with open(EXP_DIR / "step300_info.json", "w") as sf:
                json.dump(s300_info, sf, indent=2)
            model.train()
            
        if batch_idx % 200 == 0:
            elapsed = time.time() - t_train_start
            speed = total_steps / max(elapsed, 1e-5)
            eta = (total_global_steps - total_steps) / speed
            print(f"[Epoch {epoch:02d}/{NUM_EPOCHS:02d}] Step {batch_idx:04d}/{steps_per_epoch} | global={total_steps}/{total_global_steps} | loss={loss_val:.4f} | speed={speed:.2f} step/s | ETA~{eta/60:.1f}m", flush=True)
            
    mean_train_loss = total_epoch_loss / steps_per_epoch
    
    # Per-epoch Dev Evaluation
    print(f"\n[VAL EVALUATION] Evaluating Epoch {epoch} on Dev set...", flush=True)
    y_true_dev, y_pred_dev, _ = predict_epoch_loader(model, dev_loader, device, None, None)
    
    # 1. Metric on fixed 400 Model Selection documents (Patch 6: EXCLUSIVELY drives checkpoint selection)
    sel_metrics = compute_all_metrics(y_true_dev[model_sel_ids], y_pred_dev[model_sel_ids], [1, 3, 5], threshold=0.5)
    sel_p1 = sel_metrics.get("P@1", 0.0)
    
    # 2. Metric on 200 continuity Diagnostic documents
    diag_metrics = compute_all_metrics(y_true_dev[diag_ids], y_pred_dev[diag_ids], [1, 3, 5], threshold=0.5)
    diag_p1 = diag_metrics.get("P@1", 0.0)
    
    # 3. Descriptive metrics on full 1,000 Dev documents
    dev_m = compute_all_metrics(y_true_dev, y_pred_dev, [1, 3, 5], threshold=0.5, include_f1=True)
    
    row = {
        "epoch": epoch,
        "train_loss": round(mean_train_loss, 6),
        "global_steps": total_steps,
        "val_P@1": dev_m.get("P@1", 0.0),
        "val_P@3": dev_m.get("P@3", 0.0),
        "val_P@5": dev_m.get("P@5", 0.0),
        "val_nDCG@3": dev_m.get("nDCG@3", 0.0),
        "val_nDCG@5": dev_m.get("nDCG@5", 0.0),
        "val_micro_f1": dev_m.get("micro_f1", 0.0),
        "val_macro_f1": dev_m.get("macro_f1", 0.0),
        "sel_P@1": sel_p1,
        "diag_P@1": diag_p1,
    }
    history.append(row)
    
    print(f"[VAL METRICS Epoch {epoch:02d}] train_loss={row['train_loss']:.4f} | Sel_P@1(400)={sel_p1:.4f} | Full_P@1={row['val_P@1']:.4f} | Diag_P@1={diag_p1:.4f} | Full_nDCG@3={row['val_nDCG@3']:.4f} | micro_F1={row['val_micro_f1']:.4f}", flush=True)
    
    # Append row to CSV
    with open(metrics_csv_path, "a") as f:
        f.write(f"{epoch},{row['train_loss']:.6f},{total_steps},{row['val_P@1']:.4f},{row['val_P@3']:.4f},{row['val_P@5']:.4f},{row['val_nDCG@3']:.4f},{row['val_nDCG@5']:.4f},{row['val_micro_f1']:.4f},{row['val_macro_f1']:.4f},{sel_p1:.4f},{diag_p1:.4f}\n")
        
    # Checkpoint selection strictly driven by 400 model_selection documents (Patch 6)
    if sel_p1 > best_sel_p1:
        best_sel_p1 = sel_p1
        best_epoch = epoch
        best_model_sd = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save({
            "epoch": epoch,
            "step": total_steps,
            "model_state_dict": best_model_sd,
            "variant": "R_AUTHOR",
            "selection_subset": "400_docs",
            "selection_P@1": best_sel_p1,
            "full_dev_P@1": row["val_P@1"],
            "run_id": RUN_ID,
        }, CKPT_DIR / "best_dev_model_only.pt")
        print(f"  * New best Selection P@1: {best_sel_p1:.4f} (Epoch {epoch}) -> saved best_dev_model_only.pt")
    elif best_model_sd is None:
        best_model_sd = torch.load(CKPT_DIR / "best_dev_model_only.pt", weights_only=False)["model_state_dict"]
        
    # Bounded Atomic Rolling Checkpoint (Bug 3: only latest.pt and previous.pt)
    ckpt_payload = {
        "epoch": epoch,
        "step": total_steps,
        "model_state_dict": model.state_dict(),
        "best_model_state_dict": best_model_sd,
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "variant": "R_AUTHOR",
        "run_id": RUN_ID,
        "label_order_hash": EXPECTED_LABEL_ORDER_HASH,
        "history": history,
        "best_sel_p1": best_sel_p1,
        "best_epoch": best_epoch,
        "best_step": total_steps if (epoch == best_epoch) else ckpt.get("best_step", total_steps) if "ckpt" in locals() else total_steps,
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state": torch.cuda.get_rng_state() if torch.cuda.is_available() else None,
        "numpy_rng_state": np.random.get_state(),
        "python_rng_state": random.getstate(),
        "val_metrics": dev_m,
    }
    
    tmp_latest = CKPT_DIR / "latest.pt.tmp"
    torch.save(ckpt_payload, tmp_latest)
    latest_ckpt = CKPT_DIR / "latest.pt"
    previous_ckpt = CKPT_DIR / "previous.pt"
    if latest_ckpt.exists():
        shutil.copy2(latest_ckpt, previous_ckpt)
    tmp_latest.replace(latest_ckpt)

# Save final epoch model-only checkpoint
torch.save({
    "epoch": NUM_EPOCHS,
    "step": total_steps,
    "model_state_dict": model.state_dict(),
    "variant": "R_AUTHOR",
    "final_dev_metrics": history[-1],
    "run_id": RUN_ID,
}, CKPT_DIR / "final_model_only.pt")

with open(history_json_path, "w") as f:
    json.dump(history, f, indent=2)

print(f"Training completed successfully in {(time.time() - t_train_start)/60:.1f} minutes. Best selection epoch: {best_epoch} (Selection P@1: {best_sel_p1:.4f}).", flush=True)

In [ ]:
# Cell 6: Non-Leaky HRRM Pipeline Fitting & Held-Out Evaluation
from src.lhrm_baseline.metrics import compute_per_label_metrics

print("=== Non-Leaky HRRM Pipeline Evaluation ===", flush=True)

# Patch 7: Explicitly load the SELECTED neural checkpoint (best on 400 selection docs)
selected_ckpt_path = CKPT_DIR / "best_dev_model_only.pt"
if not selected_ckpt_path.exists():
    print("Warning: best_dev_model_only.pt not found, falling back to final_model_only.pt")
    selected_ckpt_path = CKPT_DIR / "final_model_only.pt"

print(f"Loading selected neural checkpoint: {selected_ckpt_path.name}")
ckpt_sel = torch.load(selected_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt_sel["model_state_dict"])
model.eval()

# Recompute fresh predictions on the Dev set using this exact selected checkpoint
print("Recomputing fresh Dev predictions using the selected neural model...")
with torch.no_grad():
    y_true_dev_sel, y_pred_dev_sel, _ = predict_epoch_loader(model, dev_loader, device, None, None)

# Save neural predictions on full Dev
save_predictions(y_true_dev_sel, y_pred_dev_sel, label_cols, str(EXP_DIR / "neural_dev_predictions.json"))

# Bug 1 Fix: compute_per_label_metrics returns list[dict], must wrap in DataFrame for .to_csv()
per_label_rows = compute_per_label_metrics(y_true_dev_sel, y_pred_dev_sel, label_cols, threshold=0.5)
pd.DataFrame(per_label_rows).to_csv(EXP_DIR / "per_label_metrics.csv", index=False)
print(f"Saved per_label_metrics.csv ({len(per_label_rows)} labels).")

# Evaluate non-leaky HRRM pipeline using disjoint sets:
# - hrrm_fit_ids (200 docs): used EXCLUSIVELY to fit Gradient Boosting regressor
# - hrrm_eval_ids (200 docs): used EXCLUSIVELY to evaluate neural vs re-ranked HRRM
hrrm_results = evaluate_hrrm_heldout(
    y_true_full=y_true_dev_sel,
    y_pred_full=y_pred_dev_sel,
    label_order=label_cols,
    partition=dev_partition,
    seed=SEED,
)

# Export HRRM model
hrrm_model_path = EXP_DIR / "hrrm_model.joblib"
hrrm_results["hrrm_model"].save(str(hrrm_model_path))

# Save comparison JSON
comparison_record = {
    "protocol": "NON_LEAKY_DEV_PARTITION",
    "selected_checkpoint": selected_ckpt_path.name,
    "n_fit_docs": hrrm_results["n_fit_docs"],
    "n_eval_docs": hrrm_results["n_eval_docs"],
    "neural_metrics": hrrm_results["neural_metrics"],
    "hrrm_metrics": hrrm_results["hrrm_metrics"],
    "delta": hrrm_results["delta"],
}
with open(EXP_DIR / "hrrm_heldout_comparison.json", "w") as f:
    json.dump(comparison_record, f, indent=2)

print("\n--- Held-Out HRRM Comparison (Evaluated strictly on held-out 200 Dev rows) ---")
print(f"Neural Base:  P@1={hrrm_results['neural_metrics']['P@1']:.4f} | P@3={hrrm_results['neural_metrics']['P@3']:.4f} | nDCG@3={hrrm_results['neural_metrics']['nDCG@3']:.4f}")
print(f"HRRM Re-rank: P@1={hrrm_results['hrrm_metrics']['P@1']:.4f} | P@3={hrrm_results['hrrm_metrics']['P@3']:.4f} | nDCG@3={hrrm_results['hrrm_metrics']['nDCG@3']:.4f}")
print(f"Delta:        P@1={hrrm_results['delta']['P@1']:+.4f} | P@3={hrrm_results['delta']['P@3']:+.4f} | nDCG@3={hrrm_results['delta']['nDCG@3']:+.4f}")
print("Non-leaky HRRM evaluation complete.", flush=True)

In [ ]:
# Cell 7: Client Inference Demonstration with Consistent Selected Checkpoint
from IPython.display import display, Markdown

print("=== Client Inference Demonstration ===", flush=True)
print(f"Using selected neural model: {selected_ckpt_path.name} and fitted HRRM regressor.")

demo_abstracts = [
    "We present a novel neural architecture for multi-label text classification using contextual similarity attention and label embeddings. Our hierarchical ranking approach learns cross-label dependencies and optimizes ranking metrics directly.",
    "Quantum phase transitions in disordered spin systems are investigated using numerical Monte Carlo simulations. We examine the low-temperature critical exponents and spatial correlation functions across the disordered phase boundary.",
    "Efficient algorithms for convex optimization in high-dimensional machine learning problems. We analyze stochastic gradient methods with adaptive step-sizes and prove linear convergence under strong convexity assumptions."
]

demo_results = []
for i, abstract_text in enumerate(demo_abstracts, 1):
    res = predict_abstract(
        text=abstract_text,
        model=model,
        tokenizer=tokenizer,
        label_order=label_cols,
        device=device,
        hrrm_model=hrrm_results["hrrm_model"],
        top_k=5,
    )
    res["selected_checkpoint"] = selected_ckpt_path.name
    demo_results.append(res)
    print(f"\n--- Abstract Sample {i} ---")
    report_md = format_prediction_report(res)
    display(Markdown(report_md))

with open(EXP_DIR / "inference_demo.json", "w") as f:
    json.dump(demo_results, f, indent=2)

print("Inference demonstration complete.", flush=True)

In [ ]:
# Cell 8: Honest Evidence Packaging, Integrity Manifest & Downloadable Links
import zipfile
from IPython.display import FileLink

print("=== Evidence Packaging & Integrity Verification ===", flush=True)

# 1. Generate RUN_MANIFEST.json
run_manifest = {
    "run_id": RUN_ID,
    "model_variant": MODEL_VARIANT,
    "loss": LOSS_TYPE,
    "learning_rate": LR,
    "batch_size": BATCH_SIZE,
    "max_len": MAX_LEN,
    "num_epochs": NUM_EPOCHS,
    "seed": SEED,
    "bundle_name": BUNDLE_NAME,
    "bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "internal_manifest_sha256": EXPECTED_INTERNAL_MANIFEST_SHA256,
    "label_order_hash": EXPECTED_LABEL_ORDER_HASH,
    "selected_checkpoint": selected_ckpt_path.name,
    "best_selection_p1": best_sel_p1,
    "best_epoch": best_epoch,
    "total_global_steps": total_steps,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(EXP_DIR / "RUN_MANIFEST.json", "w", encoding="utf-8") as rf:
    json.dump(run_manifest, rf, indent=2)

# 2. Generate training_summary.json
training_summary = {
    "run_id": RUN_ID,
    "status": "COMPLETED",
    "total_epochs": NUM_EPOCHS,
    "total_steps": total_steps,
    "best_selection_epoch": best_epoch,
    "best_selection_p1": best_sel_p1,
    "final_epoch_metrics": history[-1] if history else {},
    "hrrm_heldout_delta": hrrm_results.get("delta", {}),
    "dev_partition_sha256": partition_checksum,
}
with open(EXP_DIR / "training_summary.json", "w", encoding="utf-8") as tf:
    json.dump(training_summary, tf, indent=2)

# 3. Fail-Closed Verification of Mandatory Evidence Files (Bug 4)
mandatory_evidence_files = [
    "RUN_MANIFEST.json",
    "CONFIG_RESOLVED.json",
    "dev_partition.json",
    "epoch_metrics.csv",
    "train_history.json",
    "training_summary.json",
    "per_label_metrics.csv",
    "step300_info.json",
    "hrrm_heldout_comparison.json",
    "inference_demo.json",
    "neural_dev_predictions.json",
]

missing_evidence = [f for f in mandatory_evidence_files if not (EXP_DIR / f).exists()]
if missing_evidence:
    raise RuntimeError(
        f"FAIL-CLOSED GATE: Missing mandatory evidence files: {missing_evidence} in {EXP_DIR}! "
        "Cannot package incomplete EVIDENCE.zip."
    )

# 4. Package into EVIDENCE.zip with SHA256SUMS.txt
evidence_zip_path = Path("/kaggle/working/EVIDENCE.zip")
sha256_records = []
with zipfile.ZipFile(evidence_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as ez:
    for fname in mandatory_evidence_files:
        fpath = EXP_DIR / fname
        h = hashlib.sha256(fpath.read_bytes()).hexdigest()
        sha256_records.append(f"{h}  {fname}")
        ez.write(fpath, arcname=fname)
        print(f"Packed: {fname} ({h[:12]}...)")
            
    sha_str = "\n".join(sha256_records) + "\n"
    ez.writestr("SHA256SUMS.txt", sha_str)
    with open(EXP_DIR / "SHA256SUMS.txt", "w") as sf:
        sf.write(sha_str)

print(f"\nEVIDENCE.zip created: {evidence_zip_path.stat().st_size:,} bytes")
ez_hash = hashlib.sha256(evidence_zip_path.read_bytes()).hexdigest()
print(f"EVIDENCE.zip SHA-256: {ez_hash}")

# 5. Copy key downloadable deliverables into /kaggle/working/
shutil.copy2(selected_ckpt_path, Path("/kaggle/working") / selected_ckpt_path.name)
if (CKPT_DIR / "final_model_only.pt").exists() and selected_ckpt_path.name != "final_model_only.pt":
    shutil.copy2(CKPT_DIR / "final_model_only.pt", Path("/kaggle/working/final_model_only.pt"))
shutil.copy2(EXP_DIR / "epoch_metrics.csv", Path("/kaggle/working/epoch_metrics.csv"))
shutil.copy2(EXP_DIR / "training_summary.json", Path("/kaggle/working/training_summary.json"))
shutil.copy2(EXP_DIR / "hrrm_heldout_comparison.json", Path("/kaggle/working/hrrm_heldout_comparison.json"))

print("\n=== Downloadable Artifacts (Click to Download) ===")
display(FileLink("EVIDENCE.zip"))
display(FileLink("epoch_metrics.csv"))
display(FileLink("training_summary.json"))
display(FileLink("hrrm_heldout_comparison.json"))
display(FileLink(selected_ckpt_path.name))
if (Path("/kaggle/working/final_model_only.pt")).exists():
    display(FileLink("final_model_only.pt"))
print("All client deliverables ready.", flush=True)